# RAG Retrieval Metrics Evaluation

## Setup
Run this first to generate gRPC stubs:
```bash
pip install grpcio grpcio-tools
cd rag/pkg/rag && python -m grpc_tools.protoc -I=../../api --python_out=. --grpc_python_out=. ../../api/rag.proto
```

In [1]:
import json
from tqdm import tqdm
import grpc
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Set
from datetime import datetime
import sys

# Add path to generated proto files
sys.path.insert(0, '../pkg/rag')

import rag_pb2
import rag_pb2_grpc

## Configuration

In [2]:
# RAG Service connection
RAG_HOST = 'localhost'
RAG_PORT = 50051

# Load evaluation dataset
DATASET_PATH = '../rag_eval_dataset.json'

# Comparison methods enum mapping
COMPARISON_METHODS = {
    'COSINE': rag_pb2.COMPARISON_METHOD_COSINE,
    'DOT': rag_pb2.COMPARISON_METHOD_DOT,
    'EUCLIDEAN': rag_pb2.COMPARISON_METHOD_EUCLIDEAN,
    'L1': rag_pb2.COMPARISON_METHOD_L1
}

# Parameter grid to test
PARAM_GRID = {
    'limit': [
        3,
        5,
        10,
    ],
    'similarity_threshold': [
        0.33,
        0.5,
        0.75,
    ],
    'comparison_method': ['COSINE', 'DOT', 'EUCLIDEAN', 'L1']
}

# Metric N values (cutoffs for P@N, R@N)
N_VALUES = [3, 5, 10]

## Load Dataset

In [3]:
# with open(DATASET_PATH, 'r', encoding='utf-8') as f:
#     dataset = json.load(f)

# print(f"Loaded {len(dataset)} queries")

## RAG Client

In [4]:
# Title to doc_id mapping from database
TITLE_TO_DOC_ID = {
    "Альфа стипендия": "822bd8de-7ee2-44d2-a1fa-eab1d20c08fe",
    "Базовая стипендия (ГАС)": "c4c3b273-4c8d-4fa7-bb21-260afa5e599d",
    "Вид магистратуры и совмещение с работой": "6af3bc02-9d10-4e7d-a341-7d019a81bd38",
    "Виды стипендий": "94acc660-fe66-40f0-bc7b-295ffca0a6d4",
    "ВКР как арт-проект": "1a87c462-9ec7-44ae-8820-c15610344963",
    "ВКР как бизнес-проект": "16c13543-eb3d-4c6b-9fc5-960bd99b9dd9",
    "ВКР как научная статья": "8e84ce13-143a-47c0-ac5c-b21cda706135",
    "вопросы к экзамену пирсии.pdf": "8e650961-a113-4203-b3d6-ca83fa14602d",
    "Для народов с севера, Сибири, ДВ": "f015ffe6-d252-4092-b21d-945a874f8e53",
    "Доступ к громозеке (Кластеру ПИРСИИ)": "1677f1c4-c4d6-4093-bbf5-034facfaae6d",
    "Именные стипендии": "de489421-b3d7-443a-adc1-99d8b4793aba",
    "Конкурс Портфолио": "50c65b67-491a-44e7-9f06-ff9f309dfc75",
    "Контакты преподавателей": "72400b0b-50fa-44f7-8c24-3785ac75fb46",
    "КУГ 2025-2026": "5b2ce390-65ca-4034-93bb-d98920fc0108",
    "Материальная поддержка студентам и аспирантам": "fb457d25-cdd2-4737-819f-720836dd49fa",
    "Ответы на вопросы по ГИА": "63a25a2d-d378-4d83-8f62-c3b4842378f7",
    "Отчисления": "63d99656-58ee-4db8-b8cd-9eddb7615166",
    "повышенная стипендия (ПГАС)": "22e7ac20-7eaf-442a-a357-5b643987fd13",
    "Полезные материалы для студентов выпускного курса": "d9a37344-1469-484c-9b84-526f296fda7e",
    "Получение документов об окончании Университета ИТМО": "419e61f3-1854-4307-8db6-29d9cad78341",
    "ППА (пересдача)": "d62378e5-8fc1-4418-bae3-6b6840e7952d",
    "практика": "e516f9f7-a806-4a1c-8ed3-226a1ab2b275",
    "Рабочий план ПирСИИ": "3eb7f236-a9fe-4f90-88f1-bf3375e46a4e",
    "Семестровый обмен внутри РФ": "cd80703a-8aa3-4809-9872-feddea809b98",
    "Семестровый обмен за границей": "ff3b020d-f1e6-4926-a63c-e9c47f1f087e",
    "Социальная стипендия": "8c00999e-06a9-43e0-9e42-9bb9d46c5b35",
    "Социальный проект как диплом": "bf9b4320-cf7b-4e07-a406-e56b7c9d934f",
    "Стипендии Сбера": "1200170d-4eb7-4a27-88fb-601220cc2bc2",
    "Стипендия аспирантам": "ee2095f6-1095-402f-aeff-54a3ce64ebf8",
    "Стипендия Потанина": "8232082b-c65e-49ee-be55-0e1bc0ac9f57",
    "Стипендия правительства СПБ": "4bf38b27-47d6-4377-9250-c4a970fb48c5",
    "Стипендия президента для обучения за рубежом": "bcea65ff-52e2-464f-8221-25b7a9d16669",
    "Стипендия президента и правительства": "73748d2d-fce3-4f77-97f9-b79f1a97cee1",
    "Стипендия \"Система\"": "bc8bbec9-f759-416f-88a5-3c0633032fdb",
    "Темы ВКР и научные руководители": "366426c7-228a-4d48-998d-398ec892f91a",
    "учебный план": "773b23cd-faed-4dda-b828-8fe8265cc992"
}


# Title to doc_id mapping from database
TITLE_TO_DOC_ID = {
    "Альфа стипендия": "822bd8de-7ee2-44d2-a1fa-eab1d20c08fe",
    "Базовая стипендия (ГАС)": "c4c3b273-4c8d-4fa7-bb21-260afa5e599d",
    "Вид магистратуры и совмещение с работой": "6af3bc02-9d10-4e7d-a341-7d019a81bd38",
    "Виды стипендий": "94acc660-fe66-40f0-bc7b-295ffca0a6d4",
    "ВКР как арт-проект": "1a87c462-9ec7-44ae-8820-c15610344963",
    "ВКР как бизнес-проект": "16c13543-eb3d-4c6b-9fc5-960bd99b9dd9",
    "ВКР как научная статья": "8e84ce13-143a-47c0-ac5c-b21cda706135",
    "вопросы к экзамену пирсии.pdf": "8e650961-a113-4203-b3d6-ca83fa14602d",
    "Для народов с севера, Сибири, ДВ": "f015ffe6-d252-4092-b21d-945a874f8e53",
    "Доступ к громозеке (Кластеру ПИРСИИ)": "1677f1c4-c4d6-4093-bbf5-034facfaae6d",
    "Именные стипендии": "de489421-b3d7-443a-adc1-99d8b4793aba",
    "Конкурс Портфолио": "50c65b67-491a-44e7-9f06-ff9f309dfc75",
    "Контакты преподавателей": "72400b0b-50fa-44f7-8c24-3785ac75fb46",
    "КУГ 2025-2026": "5b2ce390-65ca-4034-93bb-d98920fc0108",
    "Материальная поддержка студентам и аспирантам": "fb457d25-cdd2-4737-819f-720836dd49fa",
    "Ответы на вопросы по ГИА": "63a25a2d-d378-4d83-8f62-c3b4842378f7",
    "Отчисления": "63d99656-58ee-4db8-b8cd-9eddb7615166",
    "повышенная стипендия (ПГАС)": "22e7ac20-7eaf-442a-a357-5b643987fd13",
    "Полезные материалы для студентов выпускного курса": "d9a37344-1469-484c-9b84-526f296fda7e",
    "Получение документов об окончании Университета ИТМО": "419e61f3-1854-4307-8db6-29d9cad78341",
    "ППА (пересдача)": "d62378e5-8fc1-4418-bae3-6b6840e7952d",
    "практика": "e516f9f7-a806-4a1c-8ed3-226a1ab2b275",
    "Рабочий план ПирСИИ": "3eb7f236-a9fe-4f90-88f1-bf3375e46a4e",
    "Семестровый обмен внутри РФ": "cd80703a-8aa3-4809-9872-feddea809b98",
    "Семестровый обмен за границей": "ff3b020d-f1e6-4926-a63c-e9c47f1f087e",
    "Социальная стипендия": "8c00999e-06a9-43e0-9e42-9bb9d46c5b35",
    "Социальный проект как диплом": "bf9b4320-cf7b-4e07-a406-e56b7c9d934f",
    "Стипендии Сбера": "1200170d-4eb7-4a27-88fb-601220cc2bc2",
    "Стипендия аспирантам": "ee2095f6-1095-402f-aeff-54a3ce64ebf8",
    "Стипендия Потанина": "8232082b-c65e-49ee-be55-0e1bc0ac9f57",
    "Стипендия правительства СПБ": "4bf38b27-47d6-4377-9250-c4a970fb48c5",
    "Стипендия президента для обучения за рубежом": "bcea65ff-52e2-464f-8221-25b7a9d16669",
    "Стипендия президента и правительства": "73748d2d-fce3-4f77-97f9-b79f1a97cee1",
    'Стипендия "Система"': "bc8bbec9-f759-416f-88a5-3c0633032fdb",
    "Темы ВКР и научные руководители": "366426c7-228a-4d48-998d-398ec892f91a",
    "учебный план": "773b23cd-faed-4dda-b828-8fe8265cc992"
}

class RAGClient:
    def __init__(self, host, port):
        self.channel = grpc.insecure_channel(f'{host}:{port}')
        self.stub = rag_pb2_grpc.RagServiceStub(self.channel)

    def search(self, query, limit=10, similarity_threshold=0.0, comparison_method='COSINE'):
        method = COMPARISON_METHODS.get(comparison_method, rag_pb2.COMPARISON_METHOD_COSINE)
        request = rag_pb2.SearchRequest(
            query=query,
            limit=limit,
            similarity_threshold=similarity_threshold,
            comparison_method=method
        )
        response = self.stub.SearchDocuments(request)
        results = []
        for doc in response.results:
            doc_id = doc.metadata.get('doc_id', doc.title)
            results.append({
                'doc_id': doc_id,
                'title': doc.title,
                'score': doc.similarity_score
            })
        return results

    def close(self):
        self.channel.close()

client = RAGClient(RAG_HOST, RAG_PORT)

## Metrics Calculation

In [5]:
def precision_at_k(retrieved, relevant, k):
    if k == 0 or not retrieved:
        return 0.0
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & relevant) / len(retrieved_k)

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & relevant) / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for i, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / i
    return 0.0

def calculate_metrics(retrieved, relevant, n_values):
    metrics = {}
    for n in n_values:
        metrics[f'P@{n}'] = precision_at_k(retrieved, relevant, n)
        metrics[f'R@{n}'] = recall_at_k(retrieved, relevant, n)
    metrics['MRR'] = reciprocal_rank(retrieved, relevant)
    return metrics

## Run Evaluation

In [6]:
# all_results = []

# for limit in PARAM_GRID['limit']:
#     for sim_thresh in PARAM_GRID['similarity_threshold']:
#         for method in PARAM_GRID['comparison_method']:
#             params = {
# Progress for each config
#                 'limit': limit,
#                 'similarity_threshold': sim_thresh,
#                 'comparison_method': method
#             }
#             print(f"Testing: limit={limit}, sim_thresh={sim_thresh}, method={method}")
            
#             all_metrics = defaultdict(list)
            
# #             for item in dataset:

# for item in tqdm(dataset, desc="Queries"):
#                 query = item['query']
#                 relevant_docs = set(
#                     TITLE_TO_DOC_ID.get(doc, doc) for doc in item['docs']
#                 )
                
#                 try:
#                     retrieved = client.search(
#                         query=query,
#                         limit=limit,
#                         similarity_threshold=sim_thresh,
#                         comparison_method=method
#                     )
#                     retrieved_ids = [r['doc_id'] for r in retrieved]
#                 except Exception as e:
#                     print(f"  Error: {e}")
#                     retrieved_ids = []
                
#                 metrics = calculate_metrics(retrieved_ids, relevant_docs, N_VALUES)
#                 for k, v in metrics.items():
#                     all_metrics[k].append(v)
            
#             avg_metrics = {k: sum(v) / len(v) for k, v in all_metrics.items()}
#             result_row = {
#                 'timestamp': datetime.now().isoformat(),
#                 **params,
#                 **avg_metrics,
#                 'queries_count': len(dataset)
#             }
#             all_results.append(result_row)

# results_df = pd.DataFrame(all_results)
# print("Evaluation complete!")


## Results Summary

In [7]:
# display_cols = ['limit', 'similarity_threshold', 'comparison_method', 'P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR']
# results_df[display_cols].sort_values('comparison_method', ascending=False)

In [8]:
# # Calculate combined score (F1-like balance between precision and recall)
# results_df['F1@5'] = 2 * results_df['P@5'] * results_df['R@5'] / (results_df['P@5'] + results_df['R@5'] + 1e-10)
# results_df['Combined'] = results_df['P@5'] * 0.4 + results_df['R@5'] * 0.3 + results_df['MRR'] * 0.3

# print("Best by Combined Score:")
# best_combined_idx = results_df['Combined'].idxmax()
# best_combined = results_df.loc[best_combined_idx]
# print(f"  limit: {best_combined['limit']}")
# print(f"  similarity_threshold: {best_combined['similarity_threshold']}")
# print(f"  comparison_method: {best_combined['comparison_method']}")
# print(f"\nMetrics:")
# for col in ['P@5', 'R@5', 'P@10', 'R@10', 'MRR', 'F1@5', 'Combined']:
#     print(f"  {col}: {best_combined[col]:.4f}")

# print("\n\nBest by F1@5:")
# best_f1_idx = results_df['F1@5'].idxmax()
# best_f1 = results_df.loc[best_f1_idx]
# print(f"  limit: {best_f1['limit']}")
# print(f"  similarity_threshold: {best_f1['similarity_threshold']}")
# print(f"  comparison_method: {best_f1['comparison_method']}")
# print(f"  F1@5: {best_f1['F1@5']:.4f}")

# print("\n\nBest by MRR (for comparison):")
# best_mrr_idx = results_df['MRR'].idxmax()
# best_mrr = results_df.loc[best_mrr_idx]
# print(f"  limit: {best_mrr['limit']}")
# print(f"  similarity_threshold: {best_mrr['similarity_threshold']}")
# print(f"  comparison_method: {best_mrr['comparison_method']}")
# print(f"  MRR: {best_mrr['MRR']:.4f}, P@5: {best_mrr['P@5']:.4f}")


## Save Results

In [9]:
# timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
# output_path = f'./evaluation_results_{timestamp_str}.csv'
# results_df.to_csv(output_path, index=False)
# print(f"Results saved to: {output_path}")

## Visualization

In [10]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Compare methods across limits
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# metrics_to_plot = ['P@5', 'R@5', 'MRR']
# for ax, metric in zip(axes, metrics_to_plot):
#     for method in PARAM_GRID['comparison_method']:
#         method_data = results_df[results_df['comparison_method'] == method]
#         ax.plot(method_data['limit'], method_data[metric], marker='o', label=method)
#     ax.set_xlabel('Limit')
#     ax.set_ylabel(metric)
#     ax.set_title(metric)
#     ax.legend()
#     ax.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

## Multi-Dataset Evaluation
Run evaluation across all datasets and compute weighted average metrics.

In [11]:
# Define all datasets to evaluate
DATASETS = {
    'dataset_1': '../rag_eval_dataset_1.json',
    'dataset_2': '../rag_eval_dataset_2.json',
    'dataset_3': '../rag_eval_dataset_3.json',
}

def run_evaluation_for_dataset(dataset_path, dataset_name):
    """Run evaluation on a single dataset and return results."""
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    print(f"\n=== Evaluating {dataset_name}: {len(dataset)} queries ===")

    all_results = []

    config_combinations = [
        (limit, sim_thresh, method)
        for limit in PARAM_GRID['limit']
        for sim_thresh in PARAM_GRID['similarity_threshold']
        for method in PARAM_GRID['comparison_method']
    ]

    total_configs = len(config_combinations)
    for config_idx, (limit, sim_thresh, method) in enumerate(config_combinations):
        if config_idx % 9 == 0:  # Print progress every 9 configs
            print(f"  Progress: {config_idx}/{total_configs} configs done")

        params = {
            'limit': limit,
            'similarity_threshold': sim_thresh,
            'comparison_method': method
        }

        all_metrics = defaultdict(list)

        for item_idx, item in enumerate(dataset):
            query = item['query']
            relevant_docs = set(
                TITLE_TO_DOC_ID.get(doc, doc) for doc in item['docs']
            )

            try:
                retrieved = client.search(
                    query=query,
                    limit=limit,
                    similarity_threshold=sim_thresh,
                    comparison_method=method
                )
                retrieved_ids = [r['doc_id'] for r in retrieved]
            except Exception as e:
                print(f"  Error: {e}")
                retrieved_ids = []

            metrics = calculate_metrics(retrieved_ids, relevant_docs, N_VALUES)
            for k, v in metrics.items():
                all_metrics[k].append(v)

        avg_metrics = {k: sum(v) / len(v) for k, v in all_metrics.items()}
        result_row = {
            'dataset': dataset_name,
            'queries_count': len(dataset),
            **params,
            **avg_metrics,
        }
        all_results.append(result_row)

    print(f"  Finished {total_configs} configs for {dataset_name}")
    return pd.DataFrame(all_results)

In [12]:
# Run evaluation on all datasets
all_dataset_results = []
for dataset_name, dataset_path in DATASETS.items():
    try:
        df = run_evaluation_for_dataset(dataset_path, dataset_name)
        all_dataset_results.append(df)
    except Exception as e:
        print(f"Error loading {dataset_name}: {e}")

# Combine all results
combined_df = pd.concat(all_dataset_results, ignore_index=True)
print(f"\nTotal configurations tested: {len(combined_df)}")


=== Evaluating dataset_1: 60 queries ===
  Progress: 0/36 configs done
  Progress: 9/36 configs done
  Progress: 18/36 configs done
  Progress: 27/36 configs done
  Finished 36 configs for dataset_1

=== Evaluating dataset_2: 51 queries ===
  Progress: 0/36 configs done
  Progress: 9/36 configs done
  Progress: 18/36 configs done
  Progress: 27/36 configs done
  Finished 36 configs for dataset_2

=== Evaluating dataset_3: 55 queries ===
  Progress: 0/36 configs done
  Progress: 9/36 configs done
  Progress: 18/36 configs done
  Progress: 27/36 configs done
  Finished 36 configs for dataset_3

Total configurations tested: 108


In [13]:
# Compute weighted average across datasets
# Weight by number of queries in each dataset

# Get total queries per dataset
queries_per_dataset = combined_df.groupby('dataset')['queries_count'].first().to_dict()
total_queries = sum(queries_per_dataset.values())

print("Queries per dataset:")
for ds, cnt in queries_per_dataset.items():
    print(f"  {ds}: {cnt} queries")
print(f"  Total: {total_queries} queries")

# Compute weighted average metrics
metric_cols = ['P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR']

weighted_results = []
param_cols = ['limit', 'similarity_threshold', 'comparison_method']

# Get unique parameter combinations
param_combinations = combined_df[param_cols].drop_duplicates()

for _, params in param_combinations.iterrows():
    mask = (
        (combined_df['limit'] == params['limit']) &
        (combined_df['similarity_threshold'] == params['similarity_threshold']) &
        (combined_df['comparison_method'] == params['comparison_method'])
    )
    subset = combined_df[mask]
    
    weighted_row = {
        'limit': params['limit'],
        'similarity_threshold': params['similarity_threshold'],
        'comparison_method': params['comparison_method'],
    }
    
    for metric in metric_cols:
        # Weighted average: sum(metric * queries) / total_queries
        weighted_sum = 0
        for _, row in subset.iterrows():
            ds_weight = queries_per_dataset[row['dataset']] / total_queries
            weighted_sum += row[metric] * ds_weight
        weighted_row[metric] = weighted_sum
    
    weighted_results.append(weighted_row)

weighted_df = pd.DataFrame(weighted_results)
print(f"\nComputed weighted average for {len(weighted_df)} parameter combinations")

Queries per dataset:
  dataset_1: 60 queries
  dataset_2: 51 queries
  dataset_3: 55 queries
  Total: 166 queries

Computed weighted average for 36 parameter combinations


In [14]:
# Calculate combined scores and find best configuration
weighted_df['F1@5'] = 2 * weighted_df['P@5'] * weighted_df['R@5'] / (weighted_df['P@5'] + weighted_df['R@5'] + 1e-10)
weighted_df['Combined'] = weighted_df['P@5'] * 0.4 + weighted_df['R@5'] * 0.3 + weighted_df['MRR'] * 0.3

display_cols = ['limit', 'similarity_threshold', 'comparison_method', 'P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR', 'F1@5', 'Combined']

print("=== WEIGHTED AVERAGE RESULTS ACROSS ALL DATASETS ===\n")
print(weighted_df[display_cols].sort_values("Combined", ascending=False).to_string(index=False))

# Save aggregated results to CSV
import os
output_dir = './evaluation_results'
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
weighted_df[display_cols].sort_values("Combined", ascending=False).to_csv(
    f'{output_dir}/weighted_avg_results_{timestamp}.csv', index=False
)
combined_df.to_csv(f'{output_dir}/all_results_{timestamp}.csv', index=False)
print(f"\nResults saved to {output_dir}/")
print(f"  - weighted_avg_results_{timestamp}.csv")
print(f"  - all_results_{timestamp}.csv")

=== WEIGHTED AVERAGE RESULTS ACROSS ALL DATASETS ===

 limit  similarity_threshold comparison_method      P@3      P@5     P@10      R@3      R@5     R@10      MRR     F1@5  Combined
     3                  0.50            COSINE 0.358434 0.358434 0.358434 0.790562 0.790562 0.790562 0.765060 0.493238  0.610060
     3                  0.75               DOT 0.311245 0.311245 0.311245 0.823695 0.823695 0.823695 0.790161 0.451779  0.608655
     3                  0.50               DOT 0.311245 0.311245 0.311245 0.823695 0.823695 0.823695 0.790161 0.451779  0.608655
     3                  0.33               DOT 0.311245 0.311245 0.311245 0.823695 0.823695 0.823695 0.790161 0.451779  0.608655
    10                  0.50            COSINE 0.358434 0.281225 0.227293 0.790562 0.844980 0.898193 0.783608 0.422000  0.601066
     5                  0.50            COSINE 0.358434 0.281225 0.281225 0.790562 0.844980 0.844980 0.778313 0.422000  0.599478
     3                  0.33               

In [15]:
# Compact table: best config for each metric
metrics_to_find = ['Combined', 'F1@5', 'MRR', 'P@5', 'R@5']
best_rows = []
for metric in metrics_to_find:
    best_idx = weighted_df[metric].idxmax()
    best = weighted_df.loc[best_idx]
    best_rows.append({
        'Metric': metric,
        'limit': int(best['limit']),
        'sim_thresh': best['similarity_threshold'],
        'method': best['comparison_method'],
        'Value': round(best[metric], 4)
    })

best_df = pd.DataFrame(best_rows)
print(best_df.to_string(index=False))

# Save compact results
best_df.to_csv(f'{output_dir}/best_results_{timestamp}.csv', index=False)
print(f"\nBest results saved to {output_dir}/best_results_{timestamp}.csv")

  Metric  limit  sim_thresh method  Value
Combined      3        0.50 COSINE 0.6101
    F1@5      3        0.50 COSINE 0.4932
     MRR     10        0.33    DOT 0.8086
     P@5      3        0.50 COSINE 0.3584
     R@5      5        0.33    DOT 0.8871

Best results saved to ./evaluation_results/best_results_20260511_162159.csv


In [16]:
# Summary: Best configurations
print("\n" + "="*60)
print("BEST CONFIGURATION (Weighted Average)")
print("="*60)

best_combined_idx = weighted_df['Combined'].idxmax()
best_combined = weighted_df.loc[best_combined_idx]

print(f"\nBest by Combined Score:")
print(f"  limit: {int(best_combined['limit'])}")
print(f"  similarity_threshold: {best_combined['similarity_threshold']}")
print(f"  comparison_method: {best_combined['comparison_method']}")
print(f"\nMetrics:")
for col in ['P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR', 'F1@5', 'Combined']:
    print(f"  {col}: {best_combined[col]:.4f}")

print("\n\nBest by F1@5:")
best_f1_idx = weighted_df['F1@5'].idxmax()
best_f1 = weighted_df.loc[best_f1_idx]
print(f"  limit: {int(best_f1['limit'])}, sim_thresh: {best_f1['similarity_threshold']}, method: {best_f1['comparison_method']}")
print(f"  F1@5: {best_f1['F1@5']:.4f}")

print("\n\nBest by MRR:")
best_mrr_idx = weighted_df['MRR'].idxmax()
best_mrr = weighted_df.loc[best_mrr_idx]
print(f"  limit: {int(best_mrr['limit'])}, sim_thresh: {best_mrr['similarity_threshold']}, method: {best_mrr['comparison_method']}")
print(f"  MRR: {best_mrr['MRR']:.4f}")


BEST CONFIGURATION (Weighted Average)

Best by Combined Score:
  limit: 3
  similarity_threshold: 0.5
  comparison_method: COSINE

Metrics:
  P@3: 0.3584
  P@5: 0.3584
  P@10: 0.3584
  R@3: 0.7906
  R@5: 0.7906
  R@10: 0.7906
  MRR: 0.7651
  F1@5: 0.4932
  Combined: 0.6101


Best by F1@5:
  limit: 3, sim_thresh: 0.5, method: COSINE
  F1@5: 0.4932


Best by MRR:
  limit: 10, sim_thresh: 0.33, method: DOT
  MRR: 0.8086


In [17]:
# Per-dataset breakdown
print("\n" + "="*60)
print("PER-DATASET SUMMARY")
print("="*60)

for dataset_name in DATASETS.keys():
    ds_df = combined_df[combined_df['dataset'] == dataset_name].copy()
    ds_df['F1@5'] = 2 * ds_df['P@5'] * ds_df['R@5'] / (ds_df['P@5'] + ds_df['R@5'] + 1e-10)
    
    best_idx = ds_df['F1@5'].idxmax()
    best = ds_df.loc[best_idx]
    
    print(f"\n{dataset_name} (best F1@5):")
    print(f"  limit={int(best['limit'])}, sim_thresh={best['similarity_threshold']}, method={best['comparison_method']}")
    print(f"  P@5={best['P@5']:.4f}, R@5={best['R@5']:.4f}, F1@5={best['F1@5']:.4f}, MRR={best['MRR']:.4f}")


PER-DATASET SUMMARY

dataset_1 (best F1@5):
  limit=3, sim_thresh=0.5, method=COSINE
  P@5=0.3750, R@5=0.8011, F1@5=0.5109, MRR=0.8333

dataset_2 (best F1@5):
  limit=3, sim_thresh=0.5, method=COSINE
  P@5=0.3268, R@5=0.7876, F1@5=0.4619, MRR=0.7680

dataset_3 (best F1@5):
  limit=3, sim_thresh=0.5, method=COSINE
  P@5=0.3697, R@5=0.7818, F1@5=0.5020, MRR=0.6879


In [18]:
# # Heatmap of MRR by limit and comparison_method
# pivot_mrr = results_df.pivot_table(
#     values='MRR',
#     index='comparison_method',
#     columns='limit',
#     aggfunc='mean'
# )

# plt.figure(figsize=(8, 5))
# sns.heatmap(pivot_mrr, annot=True, fmt='.3f', cmap='YlGnBu')
# plt.title('MRR by Comparison Method and Limit')
# plt.xlabel('Limit')
# plt.ylabel('Comparison Method')
# plt.tight_layout()
# plt.show()

## Close Connection

In [19]:
client.close()
print("Done!")

Done!
